In [2]:
import pandas as pd
import numpy as np

# 재현성 고정
np.random.seed(42)

# 데이터 로드
orders = pd.read_csv(r"C:\Users\jydom\OneDrive\문서\Project\manufacturing\00_data\01_raw_data\orders.csv")
process = pd.read_csv(r"C:\Users\jydom\OneDrive\문서\Project\manufacturing\00_data\01_raw_data\process.csv")

print("orders:\n", orders.head())
print("process:\n", process.head())

orders:
     order_id product_id  order_qty  order_date    due_date
0  ORD000001     PRD012         35  2025-11-19  2025-11-27
1  ORD000002     PRD002         87  2025-10-17  2025-11-04
2  ORD000003     PRD004         50  2025-08-28  2025-09-08
3  ORD000004     PRD008         65  2025-09-23  2025-10-11
4  ORD000005     PRD009         37  2025-11-02  2025-11-16
process:
   product_id process_id material_id  process_step process_name  \
0     PRD001          A       SKU84             1      Milling   
1     PRD001          A       SKU57             1      Milling   
2     PRD001          A       SKU72             1      Milling   
3     PRD001          A        SKU5             1      Milling   
4     PRD001          A       SKU49             1      Milling   

   required_material_qty  standard_cycle_time  
0                      4            70.208955  
1                      2            70.208955  
2                      5            70.208955  
3                      4            70

In [9]:
# 공정 기준 정보
process_master = (
    process[[
        "product_id",
        "process_id",
        "process_step",
        "standard_cycle_time",
        "process_name"
    ]]
    .drop_duplicates()
    .sort_values(["product_id", "process_step"])
)

In [10]:
# 공정별 설비 정의
machine_pool = {
    "A" : ["MILL_01", "MILL_02", "MILL_03", "MILL_04"],
    "B" : ["LATHE_01", "LATHE_02", "LATHE_03"],
    "C" : ["DRILL_01", "DRILL_02"],
    "D" : ["GRIND_01"],
    "E" : ["ADD_01", "ADD_02"]
}

In [11]:
# 공정별 setup_time 범위
setup_time_range = {
    "Milling" : (10, 25),
    "Lathe" : (8, 20),
    "Drilling" : (5, 15),
    "Grinding" : (5, 10),
    "Additive" : (15, 30)
}

In [20]:
# production_log 생성
rows = []

for idx, order in orders.iterrows():
    order_id = order["order_id"]
    product_id = order["product_id"]

    lot_id = f"LOT{str(idx + 1).zfill(6)}"

    product_processes = process_master[
        process_master["product_id"] == product_id
    ]

    for _, proc in product_processes.iterrows():
        process_id = proc["process_id"]
        process_name = proc["process_name"]
        standard_cycle_time = proc["standard_cycle_time"]

        # 설비 배정
        machine_id = np.random.choice(machine_pool[process_id])

        # setup_time
        low, high = setup_time_range[process_name]
        setup_time = round(np.random.uniform(low, high), 2)

        # downtime
        downtime = np.random.choice(
            [
                np.random.uniform(0, 10),
                np.random.uniform(10, 50)
            ],
            p=[0.85, 0.15]
        )
        downtime = round(downtime, 2)

        # actual_work_time
        actual_work_time = (
            standard_cycle_time * np.random.uniform(0.9, 1.2) + setup_time + downtime
        )
        actual_work_time = round(actual_work_time, 2)

        # job_status
        if downtime >= 30 or actual_work_time >= standard_cycle_time * 1.45:
            job_status = "Delayed"
        else:
            job_status = "Completed"
        
        rows.append({
            "lot_id" : lot_id,
            "process_id" : process_id,
            "order_id" : order_id,
            "product_id" : product_id,
            "machine_id" : machine_id,
            "actual_work_time" : actual_work_time,
            "setup_time" : setup_time,
            "downtime" : downtime,
            "job_status" : job_status
        })

raw_production_log = pd.DataFrame(rows)

raw_production_log.head()

,lot_id,process_id,order_id,product_id,machine_id,actual_work_time,setup_time,downtime,job_status
0,LOT000001,A,ORD000001,PRD012,MILL_04,103.70,22.84,2.78,Delayed
1,LOT000001,B,ORD000001,PRD012,LATHE_02,87.66,10.26,3.75,Completed
2,LOT000001,C,ORD000001,PRD012,DRILL_01,91.34,9.07,8.60,Completed
3,LOT000001,D,ORD000001,PRD012,GRIND_01,87.25,8.43,2.76,Completed
4,LOT000001,E,ORD000001,PRD012,ADD_01,115.64,23.28,23.18,Delayed


In [21]:
print("row 수:", len(raw_production_log))  # 25000
print("PK 중복:", raw_production_log.duplicated(["lot_id","process_id"]).sum())
print("lot 수:", raw_production_log["lot_id"].nunique())
print("order 수:", raw_production_log["order_id"].nunique())

print("\nlot별 공정 수:")
print(raw_production_log.groupby("lot_id")["process_id"].nunique().describe())

print("\n공정별 row 수:")
print(raw_production_log["process_id"].value_counts())

print("\n작업 상태:")
print(raw_production_log["job_status"].value_counts(normalize=True))

row 수: 25000
PK 중복: 0
lot 수: 5000
order 수: 5000

lot별 공정 수:
count    5000.0
mean        5.0
std         0.0
min         5.0
25%         5.0
50%         5.0
75%         5.0
max         5.0
Name: process_id, dtype: float64

공정별 row 수:
process_id
A    5000
B    5000
C    5000
D    5000
E    5000
Name: count, dtype: int64

작업 상태:
job_status
Completed    0.7254
Delayed      0.2746
Name: proportion, dtype: float64


In [22]:
# CSV로 내보내기
output_path = output_path = r"C:\Users\jydom\OneDrive\문서\Project\manufacturing\00_data\01_raw_data\production_log.csv"

try:
    raw_production_log.to_csv(output_path, index=False, encoding="utf-8-sig")
    print("process.csv 생성 완료:", raw_production_log.shape)
    
except Exception as e:
    print("생성 실패:", e)

process.csv 생성 완료: (25000, 9)
